# Portal without RAG, compared with RAG
### Annual report questions answered three ways, scored the same way

| Approach | Where the figure comes from |
|---|---|
| **LLM only** | Qwen's memory, with no documents and no internet |
| **Portal (SEC EDGAR XBRL)** | The SEC's data for one 10-K. Qwen only picks which fact to look up; code fetches it and writes the sentence |
| **RAG** | The annual report PDF, from the results of notebooks 01 to 07 |

**Keeping it fair:**
- The portal is limited to the FY2024 10-K, otherwise it could answer questions about years the report doesn't cover.
- Every approach uses the same 66 gold questions and the same scoring code, and the same Qwen model unless you change `QWEN_MODEL`.
- These questions were written from the annual report PDF, so they favour RAG. The table therefore also scores every approach on just the questions the portal can reach.
- Faithfulness isn't compared: LLM only has no evidence to check against, and the portal's sentence is built from the fact by code.

**To run:** choose a T4 GPU and enter your contact in `SEC_CONTACT`, as the SEC requires. To include RAG, use the same `DRIVE_FOLDER` as notebooks 01 to 07, then Runtime > Run all.

## Step 1: Environment setup

In [ ]:
# Colab: Runtime > Change runtime type > T4 GPU first.
!nvidia-smi -L
!pip install -q langchain-core langchain-community langchain-huggingface langchain-text-splitters 'pydantic>=2' pymupdf rapidocr onnxruntime sentence-transformers accelerate bitsandbytes pandas

In [ ]:
# @title Run options
USE_DRIVE = True  # @param {type:"boolean"}
DRIVE_FOLDER = "rag_arch"  # @param {type:"string"}
QUICK_TEST = False  # @param {type:"boolean"}

# With Drive, the PDF, VLM cache, results and scoreboard survive a disconnect and are shared by all seven notebooks.
# A quick test evaluates 6 questions and sends 3 pages to the VLM: it checks the notebook runs, so don't report its scores.
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["RAG_WORK_DIR"] = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
for name, value in {"EVAL_LIMIT": "6", "VLM_MAX_PAGES": "3"}.items():
    if QUICK_TEST:
        os.environ[name] = value
    else:
        os.environ.pop(name, None)

In [ ]:
# @title Company and SEC contact
TICKER = "CVX"  # @param {type:"string"}
FISCAL_YEAR = 2024  # @param {type:"integer"}
SEC_CONTACT = ""  # @param {type:"string"}
# The SEC asks every automated client to identify itself: enter "your name your@email". It is sent only to sec.gov.

import os
SEC_USER_AGENT = SEC_CONTACT or os.getenv("SEC_USER_AGENT", "")
ARCHITECTURE = "portal_comparison"

In [ ]:
import gc, glob, hashlib, json, os, re, shutil, statistics, time
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path
from typing import Annotated, Literal, Optional, Union

import pandas as pd
import torch
from pydantic import BaseModel, ConfigDict, Field, TypeAdapter, ValidationError, computed_field, field_validator, model_validator

class Settings(BaseModel):
    """Run settings, validated once. Environment variables override the defaults."""
    model_config = ConfigDict(frozen=True)
    work_dir: Path = Path("/content")
    qwen_model: str = "Qwen/Qwen2.5-7B-Instruct"  # "Qwen/Qwen2.5-8B-Instruct" does not exist on Hugging Face
    embedding_model: str = "BAAI/bge-small-en-v1.5"
    reranker_model: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"
    top_k: int = Field(5, ge=1, le=20, description="Evidence passages given to Qwen")
    max_context_chars: int = Field(7000, ge=500)
    eval_limit: int = Field(0, ge=0, description="0 runs every gold question")

CFG = Settings(**{field: value for field, value in {"work_dir": os.getenv("RAG_WORK_DIR"), "qwen_model": os.getenv("QWEN_MODEL"),
                                                    "eval_limit": os.getenv("EVAL_LIMIT")}.items() if value})
CFG.work_dir.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == "cuda" else 0
print(f"Device: {DEVICE}, VRAM: {VRAM_GB:.1f} GB")
if DEVICE == "cpu":
    print("WARNING: no GPU. In Colab choose Runtime > Change runtime type > T4 GPU; Qwen will not load in 4-bit without it.")

GOLD_VALID = True  # this notebook reads no PDF; the gold questions are for Chevron FY2024

## Step 2: Load Qwen in 4-bit through LangChain

In [ ]:
from functools import partial

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(CFG.qwen_model)
if "qwen3" in CFG.qwen_model.lower():  # thinking would break the JSON reply and use up the token limit
    tokenizer.apply_chat_template = partial(tokenizer.apply_chat_template, enable_thinking=False)
model = AutoModelForCausalLM.from_pretrained(CFG.qwen_model, quantization_config=quant, device_map={"": 0})  # 4-bit weights must stay on the GPU
# No repetition_penalty: it pushes the model away from digits already in the context, which corrupts copied figures.
text_pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=140, do_sample=False,
                     return_full_text=False, pad_token_id=tokenizer.eos_token_id)
for attribute in ("temperature", "top_p", "top_k"):
    setattr(model.generation_config, attribute, None)
CHAT_LLM = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=text_pipe), model_id=CFG.qwen_model, tokenizer=tokenizer)
print(f"Loaded {CFG.qwen_model}; GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## Step 3: Gold questions and scoring helpers
The same code the seven RAG notebooks use.

In [ ]:
class GoldBase(BaseModel):
    model_config = ConfigDict(frozen=True, extra="forbid")
    id: str
    question: str = Field(min_length=5)
    ground_truth: str = Field(min_length=1)
    pages: list[int] = Field(default_factory=list)

class NumericGold(GoldBase):
    type: Literal["numeric"]
    kind: Literal["table", "narrative", "chart"]
    value: str
    scale: int = Field(1, gt=0)

    @field_validator("value")
    @classmethod
    def value_is_a_number(cls, value):
        Decimal(value.replace(",", ""))
        return value

    @property
    def decimal(self):
        return Decimal(self.value.replace(",", ""))

class TextGold(GoldBase):
    type: Literal["text"]
    kind: Literal["text"]
    keywords: list[str] = Field(min_length=1)

class NegativeGold(GoldBase):
    type: Literal["negative"]
    kind: Literal["negative"]

GoldQuestion = Annotated[Union[NumericGold, TextGold, NegativeGold], Field(discriminator="type")]

GOLD_SET = TypeAdapter(list[GoldQuestion]).validate_python([
{
"id": "T01",
"type": "numeric",
"kind": "table",
"question": "What was Chevron's net income attributable to Chevron Corporation in 2024?",
"value": "17,661",
"scale": 1000000,
"pages": [
22,
27
],
"ground_truth": "17,661 million US dollars [p.22]"
},
{
"id": "T02",
"type": "numeric",
"kind": "table",
"question": "What were Chevron's sales and other operating revenues in 2023?",
"value": "196,913",
"scale": 1000000,
"pages": [
22,
27,
56
],
"ground_truth": "196,913 million US dollars [p.22]"
},
{
"id": "T03",
"type": "numeric",
"kind": "table",
"question": "What was Chevron's cash flow from operating activities in 2022?",
"value": "49,602",
"scale": 1000000,
"pages": [
22,
59
],
"ground_truth": "49,602 million US dollars [p.22]"
},
{
"id": "T04",
"type": "numeric",
"kind": "table",
"question": "What were Chevron's capital expenditures in 2024?",
"value": "16,448",
"scale": 1000000,
"pages": [
22,
59
],
"ground_truth": "16,448 million US dollars [p.22]"
},
{
"id": "T05",
"type": "numeric",
"kind": "table",
"question": "What were Chevron's total assets at year-end 2023?",
"value": "261,632",
"scale": 1000000,
"pages": [
22,
58
],
"ground_truth": "261,632 million US dollars [p.22]"
},
{
"id": "T06",
"type": "numeric",
"kind": "table",
"question": "What was Chevron Corporation stockholders' equity at year-end 2024?",
"value": "152,318",
"scale": 1000000,
"pages": [
22,
58
],
"ground_truth": "152,318 million US dollars [p.22]"
},
{
"id": "T07",
"type": "numeric",
"kind": "table",
"question": "What were diluted earnings per share in 2024?",
"value": "9.72",
"scale": 1,
"pages": [
22,
27
],
"ground_truth": "9.72 US dollars per share [p.22]"
},
{
"id": "T08",
"type": "numeric",
"kind": "table",
"question": "What were cash dividends per share in 2023?",
"value": "6.04",
"scale": 1,
"pages": [
22,
27
],
"ground_truth": "6.04 US dollars per share [p.22]"
},
{
"id": "T09",
"type": "numeric",
"kind": "table",
"question": "What was Chevron's net debt ratio in 2024?",
"value": "10.4",
"scale": 1,
"pages": [
22
],
"ground_truth": "10.4 percent [p.22]"
},
{
"id": "T10",
"type": "numeric",
"kind": "table",
"question": "What was the return on average capital employed in 2022?",
"value": "20.3",
"scale": 1,
"pages": [
22,
27
],
"ground_truth": "20.3 percent [p.22]"
},
{
"id": "T11",
"type": "numeric",
"kind": "table",
"question": "What was net production of crude oil, condensate and synthetic oil in 2024, in thousands of barrels per day?",
"value": "1,560",
"scale": 1,
"pages": [
23
],
"ground_truth": "1,560 thousand barrels per day [p.23]"
},
{
"id": "T12",
"type": "numeric",
"kind": "table",
"question": "What was net production of natural gas liquids in 2022, in thousands of barrels per day?",
"value": "279",
"scale": 1,
"pages": [
23
],
"ground_truth": "279 thousand barrels per day [p.23]"
},
{
"id": "T13",
"type": "numeric",
"kind": "table",
"question": "What was net production of natural gas in 2023, in millions of cubic feet per day?",
"value": "7,744",
"scale": 1,
"pages": [
23
],
"ground_truth": "7,744 million cubic feet per day [p.23]"
},
{
"id": "T14",
"type": "numeric",
"kind": "table",
"question": "What were net proved oil-equivalent reserves at year-end 2024, in millions of barrels?",
"value": "9,804",
"scale": 1,
"pages": [
23
],
"ground_truth": "9,804 million barrels [p.23]"
},
{
"id": "T15",
"type": "numeric",
"kind": "table",
"question": "How many employees did Chevron have at year-end 2024, excluding service station employees?",
"value": "39,742",
"scale": 1,
"pages": [
23
],
"ground_truth": "39,742 employees [p.23]"
},
{
"id": "T16",
"type": "numeric",
"kind": "table",
"question": "How many service station employees were excluded from the 2024 employee count?",
"value": "5,556",
"scale": 1,
"pages": [
23
],
"ground_truth": "5,556 service station employees [p.23]"
},
{
"id": "T17",
"type": "numeric",
"kind": "table",
"question": "What were total upstream earnings in 2024?",
"value": "18,602",
"scale": 1000000,
"pages": [
27
],
"ground_truth": "18,602 million US dollars [p.27]"
},
{
"id": "T18",
"type": "numeric",
"kind": "table",
"question": "What were United States downstream earnings in 2023?",
"value": "3,904",
"scale": 1000000,
"pages": [
27
],
"ground_truth": "3,904 million US dollars [p.27]"
},
{
"id": "T19",
"type": "numeric",
"kind": "table",
"question": "What was the All Other earnings line in 2022?",
"value": "2,974",
"scale": 1000000,
"pages": [
27
],
"ground_truth": "2,974 million US dollars (a loss) [p.27]"
},
{
"id": "T20",
"type": "numeric",
"kind": "table",
"question": "What was goodwill on the balance sheet at December 31, 2024?",
"value": "4,578",
"scale": 1000000,
"pages": [
58
],
"ground_truth": "4,578 million US dollars [p.58]"
},
{
"id": "T21",
"type": "numeric",
"kind": "table",
"question": "What were total liabilities at December 31, 2024?",
"value": "103,781",
"scale": 1000000,
"pages": [
58
],
"ground_truth": "103,781 million US dollars [p.58]"
},
{
"id": "T22",
"type": "numeric",
"kind": "table",
"question": "What was long-term debt at December 31, 2023?",
"value": "20,307",
"scale": 1000000,
"pages": [
58
],
"ground_truth": "20,307 million US dollars [p.58]"
},
{
"id": "T23",
"type": "numeric",
"kind": "table",
"question": "What was depreciation, depletion and amortization in the 2024 cash flow statement?",
"value": "17,282",
"scale": 1000000,
"pages": [
59
],
"ground_truth": "17,282 million US dollars [p.59]"
},
{
"id": "T24",
"type": "numeric",
"kind": "table",
"question": "What was net cash used for investing activities in 2023?",
"value": "15,232",
"scale": 1000000,
"pages": [
59
],
"ground_truth": "15,232 million US dollars [p.59]"
},
{
"id": "T25",
"type": "numeric",
"kind": "table",
"question": "What were total revenues and other income in 2024?",
"value": "202,792",
"scale": 1000000,
"pages": [
56
],
"ground_truth": "202,792 million US dollars [p.56]"
},
{
"id": "T26",
"type": "numeric",
"kind": "table",
"question": "What was the effective income tax rate in 2024?",
"value": "35.5",
"scale": 1,
"pages": [
79
],
"ground_truth": "35.5 percent [p.79]"
},
{
"id": "N01",
"type": "numeric",
"kind": "narrative",
"question": "What did Permian Basin production average in 2024, in net barrels of oil-equivalent per day?",
"value": "921,000",
"scale": 1,
"pages": [
7
],
"ground_truth": "921,000 net boe per day [p.7]"
},
{
"id": "N02",
"type": "numeric",
"kind": "narrative",
"question": "Up to what pressure can the Anchor project technology operate, in pounds per square inch?",
"value": "20,000",
"scale": 1,
"pages": [
2,
7
],
"ground_truth": "20,000 pounds per square inch [p.2]"
},
{
"id": "N03",
"type": "numeric",
"kind": "narrative",
"question": "How many gross wells were added to Chevron's Gulf of America production in 2024?",
"value": "17",
"scale": 1,
"pages": [
2
],
"ground_truth": "17 gross wells [p.2]"
},
{
"id": "N04",
"type": "numeric",
"kind": "narrative",
"question": "What is the design capacity of the Anchor FPU in gross barrels of oil per day?",
"value": "75,000",
"scale": 1,
"pages": [
3
],
"ground_truth": "75,000 gross barrels of oil per day [p.3]"
},
{
"id": "N05",
"type": "numeric",
"kind": "narrative",
"question": "How many thousand net barrels of oil-equivalent per day came from Chevron's Gulf of America assets in 2024?",
"value": "192",
"scale": 1,
"pages": [
2
],
"ground_truth": "192 thousand net boe per day [p.2]"
},
{
"id": "N06",
"type": "numeric",
"kind": "narrative",
"question": "What total assets figure, in billions of dollars, does the Chevron at a glance page show?",
"value": "256.9",
"scale": 1,
"pages": [
18
],
"ground_truth": "256.9 billion US dollars [p.18]"
},
{
"id": "C01",
"type": "numeric",
"kind": "chart",
"question": "In the five-year cumulative total returns graph, what was $100 invested in Chevron stock worth at the end of 2024?",
"value": "150",
"scale": 1,
"pages": [
21
],
"ground_truth": "150 US dollars [p.21]"
},
{
"id": "C02",
"type": "numeric",
"kind": "chart",
"question": "In the five-year cumulative total returns graph, what was $100 invested in the S&P 500 worth at the end of 2024?",
"value": "197",
"scale": 1,
"pages": [
21
],
"ground_truth": "197 US dollars [p.21]"
},
{
"id": "C03",
"type": "numeric",
"kind": "chart",
"question": "In the five-year cumulative total returns graph, what was $100 invested in the peer group worth at the end of 2024?",
"value": "153",
"scale": 1,
"pages": [
21
],
"ground_truth": "153 US dollars [p.21]"
},
{
"id": "C04",
"type": "numeric",
"kind": "chart",
"question": "According to the total stockholder returns bar chart, what was Chevron's 1-year annualized return as of 12/31/2024?",
"value": "1.4",
"scale": 1,
"pages": [
20
],
"ground_truth": "1.4 percent [p.20]"
},
{
"id": "C05",
"type": "numeric",
"kind": "chart",
"question": "According to the total stockholder returns bar chart, what was Chevron's 5-year annualized return as of 12/31/2024?",
"value": "8.5",
"scale": 1,
"pages": [
20
],
"ground_truth": "8.5 percent [p.20]"
},
{
"id": "C06",
"type": "numeric",
"kind": "chart",
"question": "According to the total stockholder returns bar chart, what was Chevron's 10-year annualized return as of 12/31/2024?",
"value": "7.0",
"scale": 1,
"pages": [
20
],
"ground_truth": "7.0 percent [p.20]"
},
{
"id": "C07",
"type": "numeric",
"kind": "chart",
"question": "What S&P 500 10-year return does the total stockholder returns chart mark?",
"value": "13.1",
"scale": 1,
"pages": [
20
],
"ground_truth": "13.1 percent [p.20]"
},
{
"id": "C08",
"type": "numeric",
"kind": "chart",
"question": "What compound annual growth rate does the indexed dividend growth chart show for Chevron?",
"value": "6.2",
"scale": 1,
"pages": [
20
],
"ground_truth": "6.2 percent [p.20]"
},
{
"id": "F01",
"type": "text",
"kind": "text",
"question": "Who is Chevron's Chairman of the Board and Chief Executive Officer?",
"keywords": [
"wirth"
],
"pages": [
12
],
"ground_truth": "Michael K. (Mike) Wirth [p.12]"
},
{
"id": "F02",
"type": "text",
"kind": "text",
"question": "Who is Chevron's Lead Director?",
"keywords": [
"austin"
],
"pages": [
12
],
"ground_truth": "Wanda M. Austin [p.12]"
},
{
"id": "F03",
"type": "text",
"kind": "text",
"question": "Which independent registered public accounting firm audited Chevron's financial statements?",
"keywords": [
"pricewaterhouse"
],
"pages": [
53,
55
],
"ground_truth": "PricewaterhouseCoopers LLP [p.53]"
},
{
"id": "F04",
"type": "text",
"kind": "text",
"question": "What rating does Standard and Poor's give the bonds issued or guaranteed by Chevron Corporation?",
"keywords": [
"aa-"
],
"pages": [
41
],
"ground_truth": "AA- [p.41]"
},
{
"id": "F05",
"type": "text",
"kind": "text",
"question": "Which companies are in the peer group used in the stock performance charts?",
"keywords": [
"bp",
"exxon",
"shell",
"total"
],
"pages": [
20,
21
],
"ground_truth": "BP, ExxonMobil, Shell and TotalEnergies [p.20]"
},
{
"id": "F06",
"type": "text",
"kind": "text",
"question": "Which company did Chevron agree to acquire in an all-stock merger announced in October 2023?",
"keywords": [
"hess"
],
"pages": [
95
],
"ground_truth": "Hess Corporation [p.95]"
},
{
"id": "X01",
"type": "negative",
"kind": "negative",
"question": "How much bitcoin does Chevron hold on its balance sheet?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X02",
"type": "negative",
"kind": "negative",
"question": "What was Tesla's total revenue in 2024?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X03",
"type": "negative",
"kind": "negative",
"question": "What was ExxonMobil's net income in 2024?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X04",
"type": "negative",
"kind": "negative",
"question": "What was Apple Inc.'s revenue in 2024?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X05",
"type": "negative",
"kind": "negative",
"question": "What was Chevron's net income in 2021?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X06",
"type": "negative",
"kind": "negative",
"question": "What were Chevron's sales and other operating revenues in 2020?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X07",
"type": "negative",
"kind": "negative",
"question": "What was Chevron's net income in 2025?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X08",
"type": "negative",
"kind": "negative",
"question": "What capital expenditures did Chevron report for 2018?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X09",
"type": "negative",
"kind": "negative",
"question": "What was Michael Wirth's total compensation in 2024?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X10",
"type": "negative",
"kind": "negative",
"question": "How many electric vehicle charging stations does Chevron operate?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X11",
"type": "negative",
"kind": "negative",
"question": "What percentage of Chevron's employees are women?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X12",
"type": "negative",
"kind": "negative",
"question": "How many solar panels did Chevron install in 2024?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X13",
"type": "negative",
"kind": "negative",
"question": "What was Chevron's closing share price on December 31, 2024?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X14",
"type": "negative",
"kind": "negative",
"question": "What was the gross margin on Chevron's lithium business in 2024?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X15",
"type": "negative",
"kind": "negative",
"question": "What is the home address of Chevron's Chief Financial Officer?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X16",
"type": "negative",
"kind": "negative",
"question": "What is Chevron's forecast net income for 2027?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X17",
"type": "negative",
"kind": "negative",
"question": "How many offshore wind turbines did Chevron build in 2024?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X18",
"type": "negative",
"kind": "negative",
"question": "How many retail coffee shops does Chevron own in Japan?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X19",
"type": "negative",
"kind": "negative",
"question": "What was Chevron's employee attrition rate in 2024?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X20",
"type": "negative",
"kind": "negative",
"question": "What was Shell's total production in 2024?",
"pages": [],
"ground_truth": "This report does not contain this information."
}
])
GOLD_BY_QUESTION = {row.question.strip().lower(): row for row in GOLD_SET}

from langchain_core.exceptions import OutputParserException
from langchain_core.output_parsers import PydanticOutputParser

class LLMAnswer(BaseModel):
    """The only output accepted from Qwen. Anything else fails closed."""
    in_report: bool = Field(description="True only if the evidence states the item for the asked company and year")
    answer: Optional[str] = Field(None, description="One short sentence with the figure as printed")
    cited_pages: list[int] = Field(default_factory=list, description="PDF pages from the evidence labels")

    @model_validator(mode="after")
    def answer_matches_flag(self):
        if self.in_report and not (self.answer or "").strip():
            raise ValueError("in_report is true but there is no answer")
        if not self.in_report:
            self.answer, self.cited_pages = None, []
        return self

class GateDecision(BaseModel):
    answer: Optional[str] = None
    reason: str = "ok"
    cited_pages: list[int] = Field(default_factory=list)

Status = Literal["LLM_ANSWER", "GROUND_TRUTH_FALLBACK"]
Source = Literal["llm", "gold_set", "closest_passage", "none"]

class GroundedResult(BaseModel):
    question: str
    status: Status
    final_answer: str
    final_source: Source
    llm_answer: Optional[str] = None
    llm_raw: str = ""
    withheld_reason: str = ""
    cited_pages: list[int] = Field(default_factory=list)
    retrieved_pages: list[int] = Field(default_factory=list)
    context: str = ""
    retrieval_s: float = Field(ge=0)
    total_s: float = Field(ge=0)

    @model_validator(mode="after")
    def fallback_is_never_llm(self):
        if (self.status == "LLM_ANSWER") != (self.final_source == "llm") or (self.status == "LLM_ANSWER") != bool(self.llm_answer):
            raise ValueError("status, source and llm_answer disagree")
        return self

SYSTEM_PROMPT = """You answer questions about one company's annual report using ONLY the evidence passages.

Reply with one JSON object and nothing else, with these keys:
"in_report": true or false
"answer": one short sentence, or null
"cited_pages": the page numbers from the evidence labels, such as [12]

Rules:
1. Copy figures exactly as printed, with the unit shown in the evidence. Do not calculate, round or estimate.
2. In a table, take the value from the row that matches the item, and within that row the value after the year asked about, such as 2023=388. "Group > Item" means the item belongs to that group.
3. The evidence must state the item for the same company and the same year as the question. If it does not, set "in_report" to false and "answer" to null. Never substitute another year, company or similar item.

Examples:
Evidence: [p.4] Units: Millions of dollars
Total widgets revenue: 2024=410; 2023=388
Question: What was widgets revenue in 2023?
Answer: {{"in_report": true, "answer": "Widgets revenue was $388 million in 2023.", "cited_pages": [4]}}

Evidence: [p.4] Units: Millions of dollars
Total widgets revenue: 2024=410; 2023=388
Question: What was widgets revenue in 2019?
Answer: {{"in_report": false, "answer": null, "cited_pages": []}}

Evidence: [p.9] The board includes the chief executive of Globex Inc.
Question: What was Globex Inc.'s revenue?
Answer: {{"in_report": false, "answer": null, "cited_pages": []}}"""

ANSWER_CHAIN = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT), ("human", "Evidence:\n{context}\n\nQuestion: {question}\nAnswer:"),
]) | CHAT_LLM | StrOutputParser()
ANSWER_PARSER = PydanticOutputParser(pydantic_object=LLMAnswer)

CITATION = re.compile(r"\[?\(?\b(?:p\.|pp\.|page)\s*(\d+)\)?\]?", re.I)
NUMBER = re.compile(r"(?<![\w.,])\(?\$?\(?(\d{1,3}(?:,\d{3})+(?:\.\d+)?|\d+(?:\.\d+)?)\)?\s*(%|thousand|million|billion|trillion)?", re.I)
SCALES = {"thousand": Decimal(10) ** 3, "million": Decimal(10) ** 6, "billion": Decimal(10) ** 9, "trillion": Decimal(10) ** 12}
REFUSAL_WORDS = re.compile(r"\b(does|do|did) not (contain|provide|include|mention|state|specify|disclose|show)\b|"
                           r"\bnot (provided|available|mentioned|disclosed|stated|specified|found|included)\b|\bno (information|data|mention)\b|"
                           r"\bcannot (be )?(determine|determined|answer|find)\b|\bunable to\b", re.I)

def numbers_in(text):
    """(digits as printed, value without scale, scale). Bare years and page citations are not figures."""
    found = []
    for match in NUMBER.finditer(CITATION.sub(" ", text)):
        printed = match.group(1)
        value = Decimal(printed.replace(",", ""))
        if "," not in printed and "." not in printed and 1900 <= value <= 2100 and not match.group(2):
            continue
        found.append((printed, value, SCALES.get((match.group(2) or "").lower(), Decimal(1))))
    return found

def same_figure(printed, value, scale, gold, gold_scale=Decimal(1)):
    """Exact, or a correct rounding to the printed decimals with at least 3 significant digits: $17.7 billion matches 17,661 million."""
    if value == gold:
        return True
    target = gold * gold_scale / scale
    decimals = len(printed.split(".")[1]) if "." in printed else 0
    if value == target:
        return True
    significant = len(printed.replace(",", "").replace(".", "").lstrip("0"))
    return significant >= 3 and scale != Decimal(1) and target.quantize(Decimal(1).scaleb(-decimals), rounding=ROUND_HALF_UP) == value

def figure_in_evidence(printed, value, scale, evidence_numbers):
    for e_printed, e_value, e_scale in evidence_numbers:
        if same_figure(printed, value, Decimal(1), e_value) or same_figure(printed, value, scale, e_value, e_scale):
            return True
        for power in (3, 6, 9):  # evidence in millions, answer in billions and the like
            if same_figure(printed, value, scale, e_value, Decimal(10) ** power):
                return True
    return False

QUESTION_STOPWORDS = {"what", "was", "were", "which", "who", "how", "many", "much", "did", "does", "the", "and", "for", "from", "with", "that",
                      "this", "has", "had", "its", "are", "chevron", "corporation", "company", "report", "year", "end", "per", "day", "total"}

def content_words(text):
    return {word.rstrip("s") for word in re.findall(r"[a-z]{3,}", text.lower()) if word not in QUESTION_STOPWORDS}

def table_cells(line):
    return [cell.strip() for cell in line.replace("Columns:", "").split("|") if cell.strip()]

def year_of(cell):
    years = re.findall(r"\b(?:19|20)\d{2}\b", cell)
    return years[-1] if len(years) == 1 and len(cell) < 30 else None

def in_year_column(header, cells, figure, year):
    """Values align with the header from the right. If the counts differ the row can't be checked, so it isn't blocked."""
    printed, value, scale = figure
    columns = header[next(i for i, cell in enumerate(header) if year_of(cell)):]
    values = [cell for cell in cells if re.fullmatch(r"[$(\s]*[\d,.]+[)%\s]*", cell)]
    if len(values) != len(columns):
        return True
    return any(year_of(column) == year and figure_in_evidence(printed, value, scale, numbers_in(cell))
               for column, cell in zip(columns, values))

def figure_placement(question, figure, docs):
    """Printed somewhere is not enough: the figure must sit beside the asked item and, in a table, under the asked year."""
    printed, value, scale = figure
    words = content_words(question)
    years = re.findall(r"\b(?:19|20)\d{2}\b", question)
    year = years[0] if len(set(years)) == 1 else None
    outcome = "figure_not_beside_item"
    for doc in docs:
        lines, header = doc.page_content.splitlines(), None
        for index, line in enumerate(lines):
            cells = table_cells(line)
            if sum(bool(year_of(cell)) for cell in cells) >= 2:
                header = cells
                continue
            if not figure_in_evidence(printed, value, scale, numbers_in(line)):
                continue
            nearby = " ".join(lines[max(0, index - 2):index + 2])
            if words and not content_words(nearby) & words:
                continue
            pairs = re.findall(r"\b((?:19|20)\d{2})=([^;]+)", line)
            if year and pairs and not any(y == year and figure_in_evidence(printed, value, scale, numbers_in(v)) for y, v in pairs):
                outcome = "figure_in_other_year_column"
                continue
            if year and header and len(cells) > 1 and not pairs and not in_year_column(header, cells, figure, year):
                outcome = "figure_in_other_year_column"
                continue
            return "ok"
    return outcome

def evidence_block(docs):
    parts, used = [], 0
    for doc in docs:
        part = f"[p.{doc.metadata['page']}] {doc.page_content}"
        if used + len(part) > CFG.max_context_chars:
            part = part[:max(0, CFG.max_context_chars - used)]
        if part:
            parts.append(part)
            used += len(part)
    return "\n\n---\n\n".join(parts)

def gate_answer(question, raw, docs):
    """Parse against the schema, then run deterministic checks on the result."""
    try:
        parsed = ANSWER_PARSER.parse(raw or "")
    except OutputParserException:
        return GateDecision(reason="invalid_llm_output")
    if not parsed.in_report:
        return GateDecision(reason="llm_refused")
    text = parsed.answer.strip()
    if REFUSAL_WORDS.search(text):
        return GateDecision(reason="llm_refused")
    pages = {d.metadata["page"] for d in docs}
    context = "\n".join(d.page_content for d in docs)
    for year in re.findall(r"\b(?:19|20)\d{2}\b", question):
        if year not in context:
            return GateDecision(reason=f"year_{year}_not_in_evidence")
    evidence_numbers = numbers_in(context) + numbers_in(question)
    asked = numbers_in(question)
    for figure in numbers_in(text):
        printed, value, scale = figure
        if not figure_in_evidence(printed, value, scale, evidence_numbers):
            return GateDecision(reason=f"unsupported_figure_{printed}")
        if not figure_in_evidence(printed, value, scale, asked):
            placement = figure_placement(question, figure, docs)
            if placement != "ok":
                return GateDecision(reason=placement)
    cited = sorted(set(parsed.cited_pages) | {int(n) for n in CITATION.findall(text)})
    if any(page not in pages for page in cited):
        return GateDecision(reason="citation_outside_evidence")
    if not cited:  # cite the first passage printing the answer's figure, otherwise the top passage
        figures = [p for p, _, _ in numbers_in(text)]
        cited = [next((d for d in docs if any(f in d.page_content for f in figures)), docs[0]).metadata["page"]]
    if not CITATION.search(text):
        text = f"{text.rstrip('.')} " + " ".join(f"[p.{page}]" for page in cited) + "."
    return GateDecision(answer=text, cited_pages=cited)

def ground_truth_for(question, docs):
    """Used only when the LLM gives no answer. Never counted as an LLM answer in the metrics."""
    gold = GOLD_BY_QUESTION.get(question.strip().lower()) if GOLD_VALID else None
    if gold:
        return gold.ground_truth, "gold_set"
    if docs:
        doc = docs[0]
        return f"The model gave no verified answer. Closest passage, not a confirmed answer [p.{doc.metadata['page']}]: {doc.page_content[:500]}", "closest_passage"
    return "No verified information on this was found in the report.", "none"

def ask(question):
    started = time.time()
    docs = RETRIEVER.invoke(question)
    retrieved = time.time()
    raw, decision = "", GateDecision(reason="no_evidence")
    context = evidence_block(docs)
    if docs:
        raw = ANSWER_CHAIN.invoke({"context": context, "question": question})
        decision = gate_answer(question, raw, docs)
    if decision.answer:
        final, source, status = decision.answer, "llm", "LLM_ANSWER"
    else:
        final, source = ground_truth_for(question, docs)
        status = "GROUND_TRUTH_FALLBACK"
    return GroundedResult(question=question, status=status, final_answer=final, final_source=source, llm_answer=decision.answer,
                          llm_raw=raw, withheld_reason="" if decision.answer else decision.reason, cited_pages=decision.cited_pages,
                          retrieved_pages=[d.metadata["page"] for d in docs], context=context,
                          retrieval_s=round(retrieved - started, 3), total_s=round(time.time() - started, 3))

def show(result):
    print(f"Q: {result.question}\n{result.status} ({result.final_source}): {result.final_answer}")
    if result.withheld_reason:
        print(f"   LLM gave no answer: {result.withheld_reason}")
    print(f"   retrieved pages {result.retrieved_pages}, {result.total_s:.1f}s\n")

In [ ]:
class ClaimVerdict(BaseModel):
    supported: bool

JUDGE_PARSER = PydanticOutputParser(pydantic_object=ClaimVerdict)
JUDGE_CHAIN = ChatPromptTemplate.from_messages([
    ("system", 'You check a claim against evidence. It is supported only if the evidence directly states it, including the same figure, item and year. Reply with one JSON object and nothing else: {{"supported": true}} or {{"supported": false}}.'),
    ("human", "Evidence:\n{context}\n\nClaim: {claim}\nVerdict:"),
]) | CHAT_LLM | StrOutputParser()
FAITHFULNESS_TARGET = 0.98
REFUSAL_TARGET = 0.95
Rate = Annotated[float, Field(ge=0, le=1)]

class EvalRow(GroundedResult):
    id: str
    type: Literal["numeric", "text", "negative"]
    kind: str
    gold: str
    answered: bool
    correct: bool
    page_hit: Optional[bool] = None
    faithfulness: Optional[Rate] = None
    unsupported_claims: list[str] = Field(default_factory=list)

class ArchitectureMetrics(BaseModel):
    architecture: str
    questions: int = Field(gt=0)
    numeric_number_match_rate: Optional[Rate] = None
    number_match_tables: Optional[Rate] = None
    number_match_narrative: Optional[Rate] = None
    number_match_charts: Optional[Rate] = None
    text_fact_accuracy: Optional[Rate] = None
    faithfulness: Optional[Rate] = None
    negative_refusal_accuracy: Optional[Rate] = None
    false_refusal_rate: Optional[Rate] = None
    fallback_rate: Rate
    gold_page_retrieved: Optional[Rate] = None
    median_retrieval_s: float = Field(ge=0)
    median_total_s: float = Field(ge=0)

    @computed_field
    @property
    def faithfulness_pass(self) -> bool:
        return self.faithfulness is not None and self.faithfulness > FAITHFULNESS_TARGET

    @computed_field
    @property
    def negative_refusal_pass(self) -> bool:
        return self.negative_refusal_accuracy is not None and self.negative_refusal_accuracy > REFUSAL_TARGET

def number_match(result, gold):
    """Gold must appear and at most one other figure may sit beside it, so listing every column can't pass."""
    if not result.llm_answer:
        return False
    asked = {v for _, v, _ in numbers_in(gold.question)}
    target = gold.decimal
    figures = [f for f in numbers_in(result.llm_answer) if f[1] not in asked or f[1] == target]
    hit = any(same_figure(p, v, s, target, Decimal(gold.scale)) or same_figure(p, v, Decimal(1), target) for p, v, s in figures)
    return hit and len(figures) <= 2

ABBREVIATION = re.compile(r"(?:\b[A-Za-z]|\b(?:Dr|Mr|Mrs|Ms|Inc|Corp|Co|Ltd|No|St|vs|p\.l\.c))\.$")

def claims_of(answer):
    claims = []
    for piece in re.split(r"(?<=[.!?])\s+", CITATION.sub("", answer).strip()):
        if claims and ABBREVIATION.search(claims[-1]):
            claims[-1] = f"{claims[-1]} {piece}"
        else:
            claims.append(piece)
    return [c.strip() for c in claims if len(c.strip()) > 2]

def judge_claim(context, claim):
    """An unreadable verdict counts as unsupported."""
    try:
        return JUDGE_PARSER.parse(JUDGE_CHAIN.invoke({"context": context, "claim": claim})).supported
    except OutputParserException:
        return False

def faithfulness(result):
    """RAGAS-style: supported claims / claims. The judge is Qwen in a separate call; the gates never use it."""
    claims = claims_of(result.llm_answer)
    verdicts = [judge_claim(result.context, claim) for claim in claims]
    return (sum(verdicts) / len(verdicts) if verdicts else 0.0), [c for c, v in zip(claims, verdicts) if not v]

def run_evaluation():
    rows = []
    gold_rows = GOLD_SET[:CFG.eval_limit] if CFG.eval_limit else GOLD_SET
    for index, gold in enumerate(gold_rows, start=1):
        result = ask(gold.question)
        answered = bool(result.llm_answer)
        if gold.type == "numeric":
            correct = number_match(result, gold)
        elif gold.type == "negative":
            correct = not answered
        else:
            correct = answered and all(k in result.llm_answer.lower() for k in gold.keywords)
        score_value, unsupported = faithfulness(result) if answered else (None, [])
        rows.append(EvalRow(**result.model_dump(), id=gold.id, type=gold.type, kind=gold.kind, gold=gold.ground_truth, answered=answered, correct=correct,
                            page_hit=bool(set(gold.pages) & set(result.retrieved_pages)) if gold.pages else None,
                            faithfulness=score_value, unsupported_claims=unsupported))
        print(f"{index:>2}/{len(gold_rows)} {gold.id} {'ok ' if correct else 'MISS'} {result.status}: {(result.llm_answer or result.withheld_reason)[:90]}")
    return rows

def score(rows):
    frame = pd.DataFrame([row.model_dump() for row in rows])
    numeric, negative, text = (frame[frame.type == t] for t in ("numeric", "negative", "text"))
    answerable = frame[frame.type != "negative"]
    answered = frame[frame.answered]
    def rate(series):
        return float(series.mean()) if len(series) else None
    return frame, ArchitectureMetrics(
        architecture=ARCHITECTURE,
        questions=len(frame),
        numeric_number_match_rate=rate(numeric.correct),
        number_match_tables=rate(numeric[numeric.kind == "table"].correct),
        number_match_narrative=rate(numeric[numeric.kind == "narrative"].correct),
        number_match_charts=rate(numeric[numeric.kind == "chart"].correct),
        text_fact_accuracy=rate(text.correct),
        faithfulness=rate(answered.faithfulness.dropna()),
        negative_refusal_accuracy=rate(negative.correct),
        false_refusal_rate=rate(~answerable.answered),
        fallback_rate=rate(frame.status == "GROUND_TRUTH_FALLBACK"),
        gold_page_retrieved=rate(answerable.page_hit.dropna().astype(bool)),
        median_retrieval_s=float(frame.retrieval_s.median()),
        median_total_s=float(frame.total_s.median()))

## Step 4: Fetch the company's filings and facts from SEC EDGAR
`submissions` lists the 10-K and annual report filings, and `companyfacts` holds the tagged XBRL figures. Downloads are saved and reused.

In [ ]:
import urllib.request
from datetime import date

SEC_CACHE = Path(os.getenv("SEC_CACHE_DIR") or CFG.work_dir / "sec_cache")
KNOWN_CIKS = {"CVX": "0000093410"}

class Filing(BaseModel):
    form: str
    accession: str
    report_date: str
    filing_date: str
    primary_document: str
    cik: str

    @computed_field
    @property
    def url(self) -> str:
        return f"https://www.sec.gov/Archives/edgar/data/{int(self.cik)}/{self.accession.replace('-', '')}/{self.primary_document}"

class Fact(BaseModel):
    concept: str
    label: str
    description: str = ""
    unit: str
    value: Decimal
    start: Optional[str] = None
    end: str

def sec_download(url, target):
    """Reuse a saved copy if there is one. SEC allows 10 requests a second; this makes very few."""
    if target.exists():
        return target
    if "@" not in SEC_USER_AGENT:
        raise ValueError(f"{target.name} isn't saved yet, so it must be fetched: set SEC_CONTACT to 'your name your@email'.")
    request = urllib.request.Request(url, headers={"User-Agent": SEC_USER_AGENT})
    with urllib.request.urlopen(request, timeout=60) as response:
        body = response.read()
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(body)
    time.sleep(0.2)
    return target

def company_cik(ticker):
    if ticker.upper() in KNOWN_CIKS:
        return KNOWN_CIKS[ticker.upper()]
    path = sec_download("https://www.sec.gov/files/company_tickers.json", SEC_CACHE / "company_tickers.json")
    for row in json.loads(path.read_text(encoding="utf-8")).values():
        if row["ticker"].upper() == ticker.upper():
            return f"{int(row['cik_str']):010d}"
    raise ValueError(f"The SEC has no ticker {ticker}")

def annual_filings(cik):
    path = sec_download(f"https://data.sec.gov/submissions/CIK{cik}.json", SEC_CACHE / "submissions" / f"CIK{cik}.json")
    recent = json.loads(path.read_text(encoding="utf-8"))["filings"]["recent"]
    rows = zip(recent["form"], recent["accessionNumber"], recent["reportDate"], recent["filingDate"], recent["primaryDocument"])
    return [Filing(form=f, accession=a, report_date=r, filing_date=d, primary_document=p, cik=cik) for f, a, r, d, p in rows if f in ("10-K", "ARS")]

def filing_facts(cik, accession):
    """Annual and year-end facts from one filing only, so the portal can't answer beyond what that filing reports."""
    path = sec_download(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json", SEC_CACHE / "companyfacts" / f"CIK{cik}.json")
    data = json.loads(path.read_text(encoding="utf-8"))
    facts = {}
    for taxonomy in data["facts"].values():
        for concept, body in taxonomy.items():
            for unit, values in body["units"].items():
                for row in values:
                    if row["accn"] != accession or (row["end"][:4] == "2100"):
                        continue
                    start = row.get("start")
                    if start and (date.fromisoformat(row["end"]) - date.fromisoformat(start)).days < 330:
                        continue
                    facts.setdefault((concept, row["end"][:4]), Fact(concept=concept, label=body.get("label") or concept, description=body.get("description") or "", unit=unit,
                                                                     value=Decimal(str(row["val"])), start=start, end=row["end"]))
    return data.get("entityName", ""), facts

CIK = company_cik(TICKER)
FILINGS = annual_filings(CIK)
print(pd.DataFrame([f.model_dump() for f in FILINGS])[["form", "report_date", "filing_date", "url"]].head(8).to_string(index=False))
TARGET = next(f for f in FILINGS if f.form == "10-K" and f.report_date.startswith(str(FISCAL_YEAR)))
ENTITY, FACTS = filing_facts(CIK, TARGET.accession)
print(f"\n{ENTITY}: {len(FACTS)} annual facts in the FY{FISCAL_YEAR} 10-K, accession {TARGET.accession}")

if "@" in SEC_USER_AGENT:  # the filings themselves, to read or to index later; the comparison below doesn't need them
    for filing in [TARGET, *[f for f in FILINGS if f.form == "ARS" and f.report_date.startswith(str(FISCAL_YEAR))]]:
        print("saved", sec_download(filing.url, CFG.work_dir / "filings" / filing.primary_document))

## Step 5: Answer without RAG
In the portal approach Qwen chooses a concept and code does the lookup, so a wrong answer means the wrong concept was chosen, not that a figure was invented.

In [ ]:
class FactRoute(BaseModel):
    """Qwen only picks what to look up. The figure always comes from the SEC data."""
    company: str = ""
    concept: Optional[str] = None
    year: Optional[int] = None

ROUTE_PROMPT = """You choose which SEC XBRL fact answers a question about {entity}.

Reply with one JSON object and nothing else, with these keys:
"company": the company the question asks about
"concept": the exact concept name from the list that states what was asked, or null if none does
"year": the year asked about, or null if none is given

Choose null unless the concept means the same thing as the question. A similar concept is not enough.

Examples:
Question: What was Globex's revenue in 2023?
Concepts: Revenues: Revenues | Assets: Assets
Answer: {{"company": "Globex", "concept": "Revenues", "year": 2023}}

Question: How many trees did Globex plant?
Concepts: Assets: Assets | Revenues: Revenues
Answer: {{"company": "Globex", "concept": null, "year": null}}"""

ROUTE_CHAIN = ChatPromptTemplate.from_messages([
    ("system", ROUTE_PROMPT), ("human", "Question: {question}\nConcepts: {concepts}\nAnswer:"),
]) | CHAT_LLM | StrOutputParser()
ROUTE_PARSER = PydanticOutputParser(pydantic_object=FactRoute)

LLM_ONLY_PROMPT = """You answer questions about companies from your own knowledge. You have no documents and no internet access.

Reply with one JSON object and nothing else, with these keys:
"in_report": true if you can answer, false if you can't
"answer": one short sentence with the figure, or null
"cited_pages": []"""

LLM_ONLY_CHAIN = ChatPromptTemplate.from_messages([
    ("system", LLM_ONLY_PROMPT), ("human", "Question: {question}\nAnswer:"),
]) | CHAT_LLM | StrOutputParser()

CATALOG = {fact.concept: fact.label for fact in FACTS.values()}
DESCRIPTIONS = {fact.concept: content_words(fact.description) for fact in FACTS.values()}

def concept_words(concept, label):
    return content_words(re.sub(r"(?<=[a-z])(?=[A-Z])", " ", concept) + " " + label)

def shortlist(question, size=15):
    """Over 300 concepts won't fit in a useful prompt. Prefer words in the name, then in the SEC's description, then short general names."""
    words = content_words(question)
    scored = [(3 * len(concept_words(c, l) & words) + len(DESCRIPTIONS[c] & words), c) for c, l in CATALOG.items()]
    return [c for score, c in sorted(scored, key=lambda pair: (-pair[0], len(pair[1]), pair[1])) if score][:size]

def format_fact(fact):
    value = fact.value
    if fact.unit == "USD":
        millions = value / Decimal(10) ** 6
        text = f"${abs(millions):,.0f} million" if abs(value) >= 10 ** 6 else f"${abs(value):,.2f}"
    elif fact.unit == "USD/shares":
        text = f"${abs(value):,.2f} per share"
    elif fact.unit == "pure":
        text = f"{abs(value) * 100:,.1f}%"
    else:
        text = f"{abs(value):,.0f} {fact.unit}"
    return ("-" if value < 0 else "") + text

def build_result(question, started, answer, raw, reason, context=""):
    if answer:
        final, source, status = answer, "llm", "LLM_ANSWER"
    else:
        final, source = ground_truth_for(question, [])
        status = "GROUND_TRUTH_FALLBACK"
    return GroundedResult(question=question, status=status, final_answer=final, final_source=source, llm_answer=answer, llm_raw=raw,
                          withheld_reason="" if answer else reason, context=context, retrieval_s=0, total_s=round(time.time() - started, 3))

def ask_llm_only(question):
    """No evidence and so no gates beyond the schema: this is what the model does when left alone."""
    started = time.time()
    raw = LLM_ONLY_CHAIN.invoke({"question": question})
    try:
        parsed = ANSWER_PARSER.parse(raw)
    except OutputParserException:
        return build_result(question, started, None, raw, "invalid_llm_output")
    if not parsed.in_report or REFUSAL_WORDS.search(parsed.answer):
        return build_result(question, started, None, raw, "llm_refused")
    return build_result(question, started, parsed.answer.strip(), raw, "")

def ask_portal(question):
    """Qwen picks a concept, code fetches the fact and writes the sentence, so no figure can come from the model."""
    started = time.time()
    candidates = shortlist(question)
    if not candidates:
        return build_result(question, started, None, "", "no_matching_concept")
    listing = " | ".join(f"{c}: {CATALOG[c]}" for c in candidates)
    raw = ROUTE_CHAIN.invoke({"entity": ENTITY, "question": question, "concepts": listing})
    try:
        route = ROUTE_PARSER.parse(raw)
    except OutputParserException:
        return build_result(question, started, None, raw, "invalid_llm_output")
    if ENTITY.split()[0].lower() not in route.company.lower():
        return build_result(question, started, None, raw, "other_company")
    if not route.concept:
        return build_result(question, started, None, raw, "llm_refused")
    # Qwen sometimes replies with the label, or "name: label". Accept an exact match on either, and nothing looser.
    named = {name.lower(): c for c in candidates for name in (c, CATALOG[c], f"{c}: {CATALOG[c]}")}
    concept = named.get(route.concept.strip().lower())
    if concept is None:
        return build_result(question, started, None, raw, "concept_not_offered")
    year = route.year or FISCAL_YEAR
    fact = FACTS.get((concept, str(year)))
    if fact is None:
        return build_result(question, started, None, raw, f"no_fact_for_{year}")
    answer = f"{fact.label}, year ended {fact.end[:4]}: {format_fact(fact)}."
    return build_result(question, started, answer, raw, "", context=f"SEC XBRL {fact.concept}, accession {TARGET.accession}, period end {fact.end}")

for question in ["What were Chevron's total assets at year-end 2023?", "What was net production of natural gas in 2023?", "How much bitcoin does Chevron hold on its balance sheet?"]:
    show(ask_portal(question))

## Step 6: Evaluate LLM only and the portal

In [ ]:
def evaluate(mode, ask_fn):
    global ARCHITECTURE
    ARCHITECTURE = mode
    rows = []
    gold_rows = GOLD_SET[:CFG.eval_limit] if CFG.eval_limit else GOLD_SET
    for index, gold in enumerate(gold_rows, start=1):
        result = ask_fn(gold.question)
        answered = bool(result.llm_answer)
        if gold.type == "numeric":
            correct = number_match(result, gold)
        elif gold.type == "negative":
            correct = not answered
        else:
            correct = answered and all(k in result.llm_answer.lower() for k in gold.keywords)
        rows.append(EvalRow(**result.model_dump(), id=gold.id, type=gold.type, kind=gold.kind, gold=gold.ground_truth, answered=answered, correct=correct))
        print(f"{index:>2}/{len(gold_rows)} {gold.id} {'ok ' if correct else 'MISS'} {(result.llm_answer or result.withheld_reason)[:90]}")
    frame, metrics = score(rows)
    frame.drop(columns=["context"]).to_csv(CFG.work_dir / f"results_{mode}.csv", index=False)
    return frame

if TICKER == "CVX" and FISCAL_YEAR == 2024:
    LLM_ONLY_RESULTS = evaluate("llm_only", ask_llm_only)
    PORTAL_RESULTS = evaluate("portal_xbrl", ask_portal)
else:
    print("The gold questions only cover Chevron FY2024, so the evaluation is skipped for", TICKER, FISCAL_YEAR)

## Step 7: Compare with RAG
- **Number match:** the share of numeric questions answered with the gold figure.
- **Portal's questions:** the same score on only the questions whose figure is in the 10-K's XBRL. Use it to compare accuracy where the portal has the data.
- **Wrong when answering:** of the answers given to answerable questions, the share that were wrong. This is the risk of trusting an answer.
- **False refusals:** answerable questions that got no answer.

In [ ]:
def in_filing(gold):
    """Could the portal answer this at all? True if the gold figure is among the filing's facts."""
    target = gold.decimal * gold.scale
    return any(abs(f.value) == target or (f.unit == "pure" and f.value * 100 == gold.decimal) for f in FACTS.values())

REACHABLE = {g.id for g in GOLD_SET if g.type == "numeric" and in_filing(g)}

def summarise(path):
    data = pd.read_csv(path)
    numeric, negative, text = (data[data.type == t] for t in ("numeric", "negative", "text"))
    answerable, answered = data[data.type != "negative"], data[(data.type != "negative") & data.answered]
    share = lambda frame: frame.correct.mean() if len(frame) else float("nan")
    return {"approach": path.stem.replace("results_", ""), "questions": len(data), "number match": share(numeric),
            "tables": share(numeric[numeric.kind == "table"]), "narrative": share(numeric[numeric.kind == "narrative"]), "charts": share(numeric[numeric.kind == "chart"]),
            f"portal's {len(REACHABLE)} questions": share(numeric[numeric.id.isin(REACHABLE)]), "text facts": share(text),
            "negative refusal": share(negative), "false refusals": (~answerable.answered).mean(),
            "wrong when answering": (~answered.correct).mean() if len(answered) else float("nan"), "median s": data.total_s.median()}

paths = sorted(CFG.work_dir.glob("results_*.csv"))
COMPARISON = pd.DataFrame([summarise(p) for p in paths]).set_index("approach")
COMPARISON.to_csv(CFG.work_dir / "portal_vs_rag_comparison.csv")
print(f"The portal can reach {len(REACHABLE)} of {sum(g.type == 'numeric' for g in GOLD_SET)} numeric questions: {sorted(REACHABLE)}\n")
display_table = COMPARISON.copy()
for column in display_table.columns:
    if column not in ("questions", "median s"):
        display_table[column] = display_table[column].map(lambda x: "n/a" if pd.isna(x) else f"{x:.1%}")
print(display_table.T.to_string())
if not any(p.stem not in ("results_llm_only", "results_portal_xbrl") for p in paths):
    print("\nNo RAG results here yet. Run notebooks 01 to 07 with the same work folder, then rerun this cell.")

portal = pd.read_csv(CFG.work_dir / "results_portal_xbrl.csv").set_index("id")
for path in paths:
    if path.stem in ("results_llm_only", "results_portal_xbrl"):
        continue
    other = pd.read_csv(path).set_index("id")
    shared = portal.index.intersection(other.index)
    print(f"\n{path.stem.replace('results_', '')}: portal right and RAG wrong {sorted(i for i in shared if portal.correct[i] and not other.correct[i])}"
          f"\n  RAG right and portal wrong {sorted(i for i in shared if other.correct[i] and not portal.correct[i] and other.type[i] != 'negative')}")

## Step 8: Download the results

In [ ]:
try:
    from google.colab import files
    for name in ["portal_vs_rag_comparison.csv", "results_llm_only.csv", "results_portal_xbrl.csv"]:
        if (CFG.work_dir / name).exists():
            files.download(str(CFG.work_dir / name))
except ImportError:
    print("Results are in", CFG.work_dir)